In [1]:
import teehr
from teehr.evaluation.spark_session_utils import create_spark_session

import os
import shutil
from pathlib import Path
from pyspark.sql import functions as F

In [2]:
spark = create_spark_session(
    start_spark_cluster=True,
    executor_instances=64,
    executor_memory="32g",
    executor_cores=2,
    update_configs={
        "spark.sql.shuffle.partitions": "512",  # 6.2B ÷ 500 = 12.4M rows/partition
        "spark.sql.adaptive.enabled": "true",
        "spark.sql.adaptive.coalescePartitions.enabled": "true",
    },
    aws_profile="admin-user"
)

INFO:teehr.evaluation.spark_session_utils:🚀 Creating Spark session: TEEHR Evaluation
INFO:teehr.evaluation.spark_session_utils:📦 Configuring Spark cluster with container image: None
INFO:teehr.evaluation.spark_session_utils:🔍 Initial spark namespace from ENV: teehr-hub
INFO:teehr.evaluation.spark_session_utils:🔍 Connecting to Kubernetes API: https://172.20.0.1:443
INFO:teehr.evaluation.spark_session_utils:🎯 Executor namespace: teehr-hub
INFO:teehr.evaluation.spark_session_utils:🔐 Executor service account: spark (in teehr-hub)
INFO:teehr.evaluation.spark_session_utils:🔐 Using in-cluster authentication
INFO:teehr.evaluation.spark_session_utils:🔗 Setting driver host to pod IP: 10.0.8.229
INFO:teehr.evaluation.spark_session_utils:✅ Spark cluster configuration successful!
INFO:teehr.evaluation.spark_session_utils:   - Executor instances: 64
INFO:teehr.evaluation.spark_session_utils:   - Executor memory: 32g
INFO:teehr.evaluation.spark_session_utils:   - Executor cores: 2
INFO:teehr.evaluati

In [3]:
ev = teehr.RemoteReadWriteEvaluation(spark=spark)

INFO:teehr.evaluation.evaluation:Using provided Spark session.
INFO:teehr.evaluation.evaluation:Active catalog set to iceberg.


In [4]:
cache_dir = '/data/data_processing/backfill-nwm/nwm-short-teehr-warehouse/local/cache/fetching/nwm/nwm30_short_range/streamflow_hourly_inst'

In [5]:
files = list(Path(cache_dir).glob("**/*.parquet"))
total_size = sum(f.stat().st_size for f in files)
print(f"Files: {len(files)}")
print(f"Total size: {total_size / (1024**3):.2f} GB")

Files: 3697
Total size: 2.11 GB


In [6]:
nwm_sdf = ev.spark.read.parquet(cache_dir)
nwm_sdf.count()

549403776

In [7]:
nwm_sdf.selectExpr("min(value_time)").show()

+-------------------+
|    min(value_time)|
+-------------------+
|2025-11-11 01:00:00|
+-------------------+



In [8]:
%%time
ev.secondary_timeseries.load_dataframe(nwm_sdf)

INFO:teehr.evaluation.tables.base_table:Initializing Table for table: secondary_timeseries.
INFO:teehr.evaluation.tables.base_table:Loading files from iceberg.teehr.secondary_timeseries.
INFO:teehr.evaluation.read:Reading files from iceberg.teehr.secondary_timeseries.
INFO:teehr.evaluation.tables.generic_table:Getting table: secondary_timeseries.
INFO:teehr.evaluation.tables.base_table:Initializing Table for table: secondary_timeseries.
INFO:teehr.evaluation.tables.base_table:Loading files from iceberg.teehr.secondary_timeseries.
INFO:teehr.evaluation.read:Reading files from iceberg.teehr.secondary_timeseries.
INFO:teehr.evaluation.validate:Start enforcing dataframe schema.
INFO:teehr.evaluation.validate:Validating DataFrame against schema.
INFO:teehr.evaluation.validate:Enforcing foreign key constraints.
INFO:teehr.evaluation.validate:Finished enforcing dataframe schema in 81.220 seconds.
INFO:teehr.evaluation.write:Start writing to warehouse table 'secondary_timeseries'.
INFO:teehr.e

CPU times: user 437 ms, sys: 29.1 ms, total: 466 ms
Wall time: 6min 59s


In [ ]:
sdf = ev.secondary_timeseries.filter([
    "configuration_name = 'nwm30_short_range'"
]).to_sdf()

In [ ]:
sdf.count()

In [ ]:
sdf.selectExpr("min(value_time)").show()

In [ ]:
spark.stop()

#### NOTE: If you are confident the data has been loaded to the warehouse delete the local evaluation data cached at: 

In [ ]:
cache_dir